# Preprocessing of the RTS data to prepare for the Issue Index creation

In [5]:
import os
import json
from bs4 import BeautifulSoup
from datetime import datetime
import pandas as pd
from tqdm import tqdm
import csv

### Directory Structure: `/mnt/project_impresso/original/RTS`

The RTS directory contains radio content organized by program/show names. The filesystem structure is organized as follows:

```
/mnt/project_impresso/original/RTS/
│
├── News Programs (Journaux)
│   ├── j_mat/                    (morning news)  
│   │   ├── audio                       (all PM3 audio files)
│   │   ├── stt                         (all stt and xml files containing the text related to each audio file)
│   │   ├── ExpXml-20251106-122951.xml  (xml files with the metadata relating the MP3 audio and XML files for each listing)
│   │   ...  
│   │   └── ExpXml-20251113-171933.xml                   
│   ├── j_midi/                   (midday news)
│   ├── j13h/                     (1 PM news)
│   ├── j13h2/                    (1 PM news variant)
│   ├── j_soir/                   (evening news)
│   └── j_nuit/                   (night news)
│
...
│
└── Other Programs
    ├── petitdej/                 (breakfast)
    ├── ana_media/                (media analysis)
    ├── geneve_info/              (Geneva information)
    └── enquest/                  (enquête/investigation)
```

Each directory contains media files (audio recordings and associated metadata) for that specific radio program.

In [6]:
base_dir = "/mnt/project_impresso/original/RTS"

audios_subdir = 'audio'
asr_subdir = 'stt'
metadata_file_start = "ExpXml"

### Process an example of metadata xml file to extract the contents

In [7]:
example_program = "causerie_uni"

example_program_dir = os.path.join(base_dir, example_program)

ex_meta_files = [os.path.join(example_program_dir,f) for f in os.listdir(example_program_dir) if f.startswith(metadata_file_start)]
ex_meta_files

['/mnt/project_impresso/original/RTS/causerie_uni/ExpXml-20251029-131700.xml',
 '/mnt/project_impresso/original/RTS/causerie_uni/ExpXml-20251029-132007.xml']

In [8]:
with open(ex_meta_files[0], "r", encoding="utf-8") as f:
    raw_xml = f.read()

xml_doc = BeautifulSoup(raw_xml, "xml")
xml_doc

<?xml version="1.0" encoding="utf-8"?>
<DOCUMENTS><DOCUMENT><CLSID>{D2593F4E-C887-4E48-8982-5BD08BA4DAE0}</CLSID><OID>{3267F04D-657F-4DCB-BCB0-44B2B6C2682D}</OID><LOGIN/><TITLE>Adolphe Dygasinski, un Kipling avant Kipling. Causerie de Edouard Cros, privat-docent ? l'Universit? de Fribourg</TITLE><HIERARCHY OID="{3267F04D-657F-4DCB-BCB0-44B2B6C2682D}" current="*" depth="---" title="Adolphe Dygasinski, un Kipling avant Kipling. Causerie de Edouard Cros, privat-docent ? l'Universit? de Fribourg"/><SEQUENCE>1</SEQUENCE><BROADCAST>Causerie universitaire</BROADCAST><DOCUMENTTYPE>Parl?</DOCUMENTTYPE><GEOGRAPHICALDESCRIPTORS><GEOGRAPHICALDESCRIPTOR>Pologne</GEOGRAPHICALDESCRIPTOR></GEOGRAPHICALDESCRIPTORS><HIERARCHYLEVEL>Sujet</HIERARCHYLEVEL><MODIFIEDBY>albrecjo</MODIFIEDBY><MODIFIEDON>25.05.2022 03:56:53</MODIFIEDON><HISTORY>Disques 78T maison Radio-Lausanne 1466X A, 1467X A, 1466X B, 1467X B</HISTORY><PARTICIPANTS><PARTICIPANT><NAME>Cros, Edouard</NAME><FUNCTION>Conf?rencier/e</FUNCTION><RO

In [6]:
docs = xml_doc.find_all("DOCUMENT")
docs 

[<DOCUMENT><CLSID>{D2593F4E-C887-4E48-8982-5BD08BA4DAE0}</CLSID><OID>{3267F04D-657F-4DCB-BCB0-44B2B6C2682D}</OID><LOGIN/><TITLE>Adolphe Dygasinski, un Kipling avant Kipling. Causerie de Edouard Cros, privat-docent ? l'Universit? de Fribourg</TITLE><HIERARCHY OID="{3267F04D-657F-4DCB-BCB0-44B2B6C2682D}" current="*" depth="---" title="Adolphe Dygasinski, un Kipling avant Kipling. Causerie de Edouard Cros, privat-docent ? l'Universit? de Fribourg"/><SEQUENCE>1</SEQUENCE><BROADCAST>Causerie universitaire</BROADCAST><DOCUMENTTYPE>Parl?</DOCUMENTTYPE><GEOGRAPHICALDESCRIPTORS><GEOGRAPHICALDESCRIPTOR>Pologne</GEOGRAPHICALDESCRIPTOR></GEOGRAPHICALDESCRIPTORS><HIERARCHYLEVEL>Sujet</HIERARCHYLEVEL><MODIFIEDBY>albrecjo</MODIFIEDBY><MODIFIEDON>25.05.2022 03:56:53</MODIFIEDON><HISTORY>Disques 78T maison Radio-Lausanne 1466X A, 1467X A, 1466X B, 1467X B</HISTORY><PARTICIPANTS><PARTICIPANT><NAME>Cros, Edouard</NAME><FUNCTION>Conf?rencier/e</FUNCTION><ROLE>privat-docent ? l'Universit? de Fribourg</ROLE

In [7]:
for child in docs[0].find_all(recursive=False):
    print(f"name: {child.name}")
    print(f"attrs: {child.attrs}")
    print(f"text: {child.get_text(strip=True)}")

name: CLSID
attrs: {}
text: {D2593F4E-C887-4E48-8982-5BD08BA4DAE0}
name: OID
attrs: {}
text: {3267F04D-657F-4DCB-BCB0-44B2B6C2682D}
name: LOGIN
attrs: {}
text: 
name: TITLE
attrs: {}
text: Adolphe Dygasinski, un Kipling avant Kipling. Causerie de Edouard Cros, privat-docent ? l'Universit? de Fribourg
name: HIERARCHY
attrs: {'title': "Adolphe Dygasinski, un Kipling avant Kipling. Causerie de Edouard Cros, privat-docent ? l'Universit? de Fribourg", 'OID': '{3267F04D-657F-4DCB-BCB0-44B2B6C2682D}', 'depth': '---', 'current': '*'}
text: 
name: SEQUENCE
attrs: {}
text: 1
name: BROADCAST
attrs: {}
text: Causerie universitaire
name: DOCUMENTTYPE
attrs: {}
text: Parl?
name: GEOGRAPHICALDESCRIPTORS
attrs: {}
text: Pologne
name: HIERARCHYLEVEL
attrs: {}
text: Sujet
name: MODIFIEDBY
attrs: {}
text: albrecjo
name: MODIFIEDON
attrs: {}
text: 25.05.2022 03:56:53
name: HISTORY
attrs: {}
text: Disques 78T maison Radio-Lausanne 1466X A, 1467X A, 1466X B, 1467X B
name: PARTICIPANTS
attrs: {}
text: Cros, 

In [25]:
# define more intuitive names
simple_fields_renaming = {
    "CLSID": "cls_ID",
    "OID": "OID",
    "LOGIN": "login",
    "TITLE": "broadcast_episode_title",
    "SEQUENCE": "sequence", 
    "BROADCAST": "broadcast_program_name",
    "DOCUMENTTYPE": "document_type",
    "HIERARCHYLEVEL": "hierarchy_level",
    "MODIFIEDBY": "modified_by",
    "MODIFIEDON": "modified_on",
    "HISTORY": "physical_support_history",
    "PRODUCTIONTYPE": "production_type",
    "RECORDINGPLACE": "recording_place",
    "RIGHTSNOTES": "rights_notes",
    "RIGHTSSTATUS": "rights_status",
    "SERIESTITLE": "series_title",
    "SUMMARY": "content_summary",
    "WORKFLOWSTATUS": "workflow_status",
    "ASSEMBLYSTATUS": "assembly_status",
    "LIVE": "live",
    "MODULATIONTYPE": "modulation_type",
    "WORKDURATION": "work_duration",
    "WORKDURATIONCOMPL": "work_duration_compl",
}

list_fields_renaming = {
    'GEOGRAPHICALDESCRIPTORS': 'geographical_descriptors',
    'PERSONDESCRIPTORS': 'person_descriptors',
    'THEMATICALDESCRIPTORS': 'thematical_descriptors',
    'RIGHTSUSAGEPOSSIBILITIES': 'rights_usage_possibilities',
    'PROGRAMMES': 'radio_channels',
    'SUBDOMAINS': 'subdomains',
    'RECORDINGDATES': 'recording_dates',
    'FIRSTBROADCASTDATES': 'first_broadcast_dates'
}

support_keys = ['spt_clsid', 'spt_oid',
                'spt_title',
                'spt_isdigital',
                'spt_filename',
                'spt_cataloguing_status',
                'spt_source',
                'spt_unit_duration']

doc_keys = ["alias", "date_str", "stt_filename", "mp3_filenames", "stripped_OID"] + list(simple_fields_renaming.values()) + list(list_fields_renaming.values()) + ["supports", "spt_filenames", "participants"]

In [ ]:
# Parse DOCUMENT elements into structured dictionaries
def parse_docs_in_xml_v1(xml_doc, program, doc_keys=doc_keys, simple_fields_map=simple_fields_renaming, list_fields_map=list_fields_renaming):
    """
    Extract all DOCUMENT elements from XML into a list of dictionaries.
    Handles nested structures: PARTICIPANTS, GEOGRAPHICALDESCRIPTORS, 
    THEMATICALDESCRIPTORS, SUPPORTS, RECORDINGDATES, etc.
    """
    documents = []
    
    # Find all DOCUMENT elements
    doc_elements = xml_doc.find_all('DOCUMENT')
    no_filenames = 0
    
    print(f"Starting extracting {len(doc_elements)} documents for program {program}")
    for doc_elem in doc_elements:
        doc_dict = {
            "alias": program,
            "date_str": None,
            "mp3_filenames": None
        }
        
        # Extract simple text fields
        
        
        for og_field, renamed_field in simple_fields_map.items():
            elem = doc_elem.find(og_field)
            if elem:
                text = elem.get_text(strip=True)
                doc_dict[renamed_field] = text
                if og_field=='OID':
                    # the text is actually "{oid}", remove start and end brackets
                    doc_dict['stripped_OID'] = text[1:-1] if text else None
            else:
                doc_dict[renamed_field] = None
        
        # Extract PARTICIPANTS (list of dicts)
        participants = []
        for participant in doc_elem.find_all('PARTICIPANT'):
            name_elem = participant.find('NAME')
            function_elem = participant.find('FUNCTION')
            role_elem = participant.find('ROLE')
            participants.append({
                'name': name_elem.get_text(strip=True) if name_elem else None,
                'function': function_elem.get_text(strip=True) if function_elem else None,
                'role': role_elem.get_text(strip=True) if role_elem else None
            })
        if participants:
            doc_dict['participants'] = participants
        
        # Extract list fields (DESCRIPTORS, PROGRAMMES, SUBDOMAINS, etc.)
        list_fields = {
            'GEOGRAPHICALDESCRIPTORS': 'GEOGRAPHICALDESCRIPTOR',
            'PERSONDESCRIPTORS': 'PERSONDESCRIPTOR',
            'THEMATICALDESCRIPTORS': 'THEMATICALDESCRIPTOR',
            'RIGHTSUSAGEPOSSIBILITIES': 'RIGHTSUSAGEPOSSIBILITY',
            'PROGRAMMES': 'PROGRAMME',
            'SUBDOMAINS': 'SUBDOMAIN',
            'RECORDINGDATES': 'RECORDINGDATE',
            'FIRSTBROADCASTDATES': 'FIRSTBROADCASTDATE'
        }
        
        for container_name, renamed_field in list_fields_map.items():
            container = doc_elem.find(container_name)
            if container:
                doc_dict[renamed_field] = [elem.get_text(strip=True) for elem in container.find_all(list_fields[container_name])]
            else:
                # always define fields, set them to None if not defined
                doc_dict[renamed_field] = None

        date_strings = None
        # extract the date from first_braodcast_dates
        if doc_dict["first_broadcast_dates"]:
            date_strings = [d for rd in doc_dict["first_broadcast_dates"] for d in rd.split(" - ") if d != "__/__/____"]
            
        if not date_strings and doc_dict["recording_dates"]:
            print(f"Did not find any date for doc with OID {doc_dict['OID']}. Trying to use the recording date. doc_dict: {doc_dict}")
            date_strings = [d for rd in doc_dict["recording_dates"] for d in rd.split(" - ") if d != "__/__/____" and "Avant" not in d and "Apr?s" not in d]
        
        # process the dates extracted
        if not date_strings:
            print(f"WARNING! MISSING DATE FOR DOC WITH OID {doc_dict['OID']}!! \nDocument: {doc_dict}, \noriginal: {doc_elem}")
            #doc_dict['day'] = None
        else:
            # reformat each date and keep the earliest
            dates = []
            for s in date_strings:
                if "__" in s:
                    old_s = s
                    s = s.replace("__", "01")
                    print(f"The date for document with OID {doc_dict['OID']} was invalid ({old_s}) - changed it to {s}")
                if s.startswith('~'):
                    dates.append(datetime.strptime(s[1:],  "%d/%m/%Y"))
                else:   
                    dates.append(datetime.strptime(s, "%d/%m/%Y"))
                

                    
            #dates = [datetime.strptime(s[1:] if s.startswith('~') else s, "%d/%m/%Y") for s in date_strings]
            #doc_dict['year'] = min(dates).year
            #doc_dict['month'] = min(dates).month
            #doc_dict['day'] = min(dates).day
            doc_dict['date_str'] = min(dates).strftime('%d/%m/%Y')
            
        
        # Extract SUPPORTS (audio files and metadata)
        supports = []
        mp3_filenames = []
        for support in doc_elem.find_all('SUPPORT'):
            support_dict = {
                'spt_clsid': support.find('CLSID').get_text(strip=True) if support.find('CLSID') else None,
                'spt_oid': support.find('OID').get_text(strip=True) if support.find('OID') else None,
                'spt_title': support.find('TITLE').get_text(strip=True) if support.find('TITLE') else None,
                'spt_isdigital': support.find('ISDIGITAL').get_text(strip=True) if support.find('ISDIGITAL') else None,
                'spt_filename': support.find('FILENAME').get_text(strip=True) if support.find('FILENAME') else None,
                'spt_cataloguing_status': support.find('CATALOGUINGSTATUS').get_text(strip=True) if support.find('CATALOGUINGSTATUS') else None,
                'spt_source': support.find('SOURCE').get_text(strip=True) if support.find('SOURCE') else None,
                'spt_unit_duration': support.find('UNITDURATION').get_text(strip=True) if support.find('UNITDURATION') else None
            }

            if support_dict['spt_filename'] and (doc_dict['stripped_OID'] or support_dict['spt_oid']):
                #print(f"doc_dict['stripped_OID']: {doc_dict['stripped_OID']}, support_dict['filename'][:-3]: {support_dict['filename'][:-3]}")
                # if the stripped OID does not exist, find the one in the supports and strip it
                filename_oid = doc_dict['stripped_OID'].lower() if doc_dict['stripped_OID'] else support_dict['spt_oid'][1:-1].lower()
                if support_dict['spt_filename'].endswith('.wav'):
                    support_filename = support_dict['spt_filename'].replace('.wav', 'mp3')
                elif support_dict['spt_filename'].endswith('}') and '.mp3' not in support_dict['spt_filename']:
                    support_filename = support_dict['spt_filename'] + '.mp3'
                mp3_filenames.append(f"{filename_oid}_{support_filename}")
            
            supports.append(support_dict)

        if supports:
            doc_dict['supports'] = supports[0]
            doc_dict['spt_filenames'] = [s['spt_filename'] for s in supports]
            doc_dict['mp3_filenames'] = mp3_filenames if mp3_filenames else None
            if not mp3_filenames:
                no_filenames += 1
                #print(f"Document with OID {doc_dict['stripped_OID']} has no associated audio/sml files! Setting to None")

        # before adding to the list of docs, check it has all keys, and setting any missing one to None
        for k in doc_keys:
            if k not in doc_dict:
                doc_dict[k] = None
        
        documents.append(doc_dict)

    print(f"{program} - returning {len(documents)} documents, {no_filenames} are missing the audio/text files.")
    
    return documents

In [26]:
# Parse DOCUMENT elements into structured dictionaries
def parse_docs_in_xml(xml_doc, program, all_alias_stt, all_alias_audios, doc_keys=doc_keys, simple_fields_map=simple_fields_renaming, list_fields_map=list_fields_renaming):
    """
    Extract all DOCUMENT elements from XML into a list of dictionaries.
    Handles nested structures: PARTICIPANTS, GEOGRAPHICALDESCRIPTORS, 
    THEMATICALDESCRIPTORS, SUPPORTS, RECORDINGDATES, etc.
    """
    documents = []
    
    # Find all DOCUMENT elements
    doc_elements = xml_doc.find_all('DOCUMENT')
    skipped = 0
    
    print(f"\nStarting extracting {len(doc_elements)} documents for program {program}")
    for doc_idx, doc_elem in enumerate(doc_elements):
        doc_dict = {
            "alias": program,
            "date_str": None,
            "stt_filename": None,
            "mp3_filenames": None
        }
        
        # Extract simple text fields
        
        
        for og_field, renamed_field in simple_fields_map.items():
            elem = doc_elem.find(og_field)
            if elem:
                text = elem.get_text(strip=True)
                doc_dict[renamed_field] = text
                if og_field=='OID':
                    # the text is actually "{oid}", remove start and end brackets
                    doc_dict['stripped_OID'] = text[1:-1] if text else None
            else:
                doc_dict[renamed_field] = None
        
        if "stripped_OID" not in doc_dict or not doc_dict['stripped_OID']:
            print(f"Skipping document {doc_idx+1}/{len(doc_elements)} because it's missing an OID!!")
            skipped +=1
            continue


        stt_file = f"{doc_dict['stripped_OID']}_STT.xml"
        if stt_file not in all_alias_stt:
            stt_text_file = f"{doc_dict['stripped_OID']}_STT.txt"
            print(f"Skipping document {doc_idx+1}/{len(doc_elements)} with OID {doc_dict['OID']} because it's missing its XML file!! (stt exists: {stt_text_file in all_alias_stt})")
            skipped +=1
            continue
        else:
            doc_dict['stt_filename'] = stt_file

        # Extract PARTICIPANTS (list of dicts)
        participants = []
        for participant in doc_elem.find_all('PARTICIPANT'):
            name_elem = participant.find('NAME')
            function_elem = participant.find('FUNCTION')
            role_elem = participant.find('ROLE')
            participants.append({
                'name': name_elem.get_text(strip=True) if name_elem else None,
                'function': function_elem.get_text(strip=True) if function_elem else None,
                'role': role_elem.get_text(strip=True) if role_elem else None
            })
        if participants:
            doc_dict['participants'] = participants
        
        # Extract list fields (DESCRIPTORS, PROGRAMMES, SUBDOMAINS, etc.)
        list_fields = {
            'GEOGRAPHICALDESCRIPTORS': 'GEOGRAPHICALDESCRIPTOR',
            'PERSONDESCRIPTORS': 'PERSONDESCRIPTOR',
            'THEMATICALDESCRIPTORS': 'THEMATICALDESCRIPTOR',
            'RIGHTSUSAGEPOSSIBILITIES': 'RIGHTSUSAGEPOSSIBILITY',
            'PROGRAMMES': 'PROGRAMME',
            'SUBDOMAINS': 'SUBDOMAIN',
            'RECORDINGDATES': 'RECORDINGDATE',
            'FIRSTBROADCASTDATES': 'FIRSTBROADCASTDATE'
        }
        
        for container_name, renamed_field in list_fields_map.items():
            container = doc_elem.find(container_name)
            if container:
                doc_dict[renamed_field] = [elem.get_text(strip=True) for elem in container.find_all(list_fields[container_name])]
            else:
                # always define fields, set them to None if not defined
                doc_dict[renamed_field] = None

        date_strings = None
        # extract the date from first_braodcast_dates
        if doc_dict["first_broadcast_dates"]:
            date_strings = [d for rd in doc_dict["first_broadcast_dates"] for d in rd.split(" - ") if d != "__/__/____"]
            
        if not date_strings and doc_dict["recording_dates"]:
            print(f"Did not find any date for doc with OID {doc_dict['OID']}. Trying to use the recording date. doc_dict: {doc_dict}")
            date_strings = [d for rd in doc_dict["recording_dates"] for d in rd.split(" - ") if d != "__/__/____" and "Avant" not in d and "Apr?s" not in d]
        
        # process the dates extracted
        if not date_strings:
            print(f"WARNING! MISSING DATE FOR DOC WITH OID {doc_dict['OID']}!! \nDocument: {doc_dict}, \noriginal: {doc_elem}")
            #doc_dict['day'] = None
        else:
            # reformat each date and keep the earliest
            dates = []
            for s in date_strings:
                if "__" in s:
                    old_s = s
                    s = s.replace("__", "01")
                    print(f"The date for document with OID {doc_dict['OID']} was invalid ({old_s}) - changed it to {s}")
                if s.startswith('~'):
                    dates.append(datetime.strptime(s[1:],  "%d/%m/%Y"))
                else:   
                    dates.append(datetime.strptime(s, "%d/%m/%Y"))
                

                    
            #dates = [datetime.strptime(s[1:] if s.startswith('~') else s, "%d/%m/%Y") for s in date_strings]
            #doc_dict['year'] = min(dates).year
            #doc_dict['month'] = min(dates).month
            #doc_dict['day'] = min(dates).day
            doc_dict['date_str'] = min(dates).strftime('%d/%m/%Y')
            
        
        # Extract SUPPORTS (audio files and metadata)
        supports = []
        mp3_filenames = []

        for support in doc_elem.find_all('SUPPORT'):
            support_dict = {
                'spt_clsid': support.find('CLSID').get_text(strip=True) if support.find('CLSID') else None,
                'spt_oid': support.find('OID').get_text(strip=True) if support.find('OID') else None,
                'spt_title': support.find('TITLE').get_text(strip=True) if support.find('TITLE') else None,
                'spt_isdigital': support.find('ISDIGITAL').get_text(strip=True) if support.find('ISDIGITAL') else None,
                'spt_filename': support.find('FILENAME').get_text(strip=True) if support.find('FILENAME') else None,
                'spt_cataloguing_status': support.find('CATALOGUINGSTATUS').get_text(strip=True) if support.find('CATALOGUINGSTATUS') else None,
                'spt_source': support.find('SOURCE').get_text(strip=True) if support.find('SOURCE') else None,
                'spt_unit_duration': support.find('UNITDURATION').get_text(strip=True) if support.find('UNITDURATION') else None
            }

            if support_dict['spt_filename']:
                # remove '.wav' if it's in the filename
                spt_filename = support_dict['spt_filename'].replace('.wav', "")
                # populate the list of mp3 filenames with any mp3 file which has a matching filename
                mp3_filenames.extend([audio for audio in all_alias_audios if spt_filename in audio])

            supports.append(support_dict)

        if not supports or not mp3_filenames:
            print(f"Skipping document {doc_idx+1}/{len(doc_elements)} with OID {doc_dict['OID']} because it's missing its audio MP3 file!!")
            skipped +=1
            continue

        doc_dict['supports'] = supports
        doc_dict['spt_filenames'] = [s['spt_filename'] for s in supports]
        doc_dict['mp3_filenames'] = mp3_filenames

        # before adding to the list of docs, check it has all keys, and setting any missing one to None
        for k in doc_keys:
            if k not in doc_dict:
                doc_dict[k] = None
        
        documents.append(doc_dict)

    print(f"{program} - returning {len(documents)} documents, {skipped} were skipped due to missing necessary info.")
    
    return documents

In [ ]:
last_alias = None
for row in no_existing_mp3s.itertuples():
    print(row.alias, row.stripped_OID, row.spt_filenames, row.mp3_filenames)
    if row.alias!=last_alias:
        # update the values if we changed alias
        audios_dir = os.path.join(BASE_DATA_PATH, PROVIDER_NAME, row.alias, "audio")
        stt_dir = os.path.join(BASE_DATA_PATH, PROVIDER_NAME, row.alias, "stt")
        last_alias = row.alias
        print(f"Changed values of audios_dir to {audios_dir} and stt_dir to {stt_dir}. Listing their contents")
        all_audios = os.listdir(audios_dir)
        all_stt = os.listdir(stt_dir)

    #matching_stt = row.stripped_OID
    stt_file = f"{row.stripped_OID}_STT.xml"
    print(f"{row.alias}: {stt_file in all_stt} - stt_file={stt_file}")

    support_filenames = [f for f in literal_eval(row.spt_filenames) if f]
    all_valid_audios = []
    for f in support_filenames:
        if f and '.wav' in f:
            print(f"removing the .wav in {f}")
            f = f.replace(".wav", '')
        valid_audios = [audio for audio in all_audios if f in audio]
        print(f"filename {f} corresponds to audio {valid_audios}!")
        all_valid_audios.extend(valid_audios)
    
    oid_also_in = [audio for audio in all_valid_audios if row.stripped_OID.lower() in audio]
    if len(all_valid_audios)>1:
        print(f"{row.alias}: There are more than 1 audio file which matches: {all_valid_audios}")
    elif len(all_valid_audios)==1:
        print(f"{row.alias}: audio_file={all_valid_audios[0]}")
    else:
        print(f"{row.alias}: No valid audio file")

Check that the function works correctly

In [9]:

audio_files = os.listdir(os.path.join(example_program_dir, audios_subdir))
text_files = os.listdir(os.path.join(example_program_dir, asr_subdir))

# Parse all documents from the example XML
all_documents = parse_docs_in_xml(xml_doc, example_program, text_files, audio_files)

print(f"Total documents found: {len(all_documents)}")
if all_documents:
    for idx, doc in enumerate(all_documents):
        #doc = all_documents[0]
        #print(f"\nFirst document keys: {list(doc.keys())}")
        print(f"\nTitle: {doc.get('title', 'N/A')[:80]}...")
        print(f"Broadcast program name: {doc.get('broadcast_program_name', 'N/A')}")
        print(f"Extracted boradcast date: {doc['date_str']} (year {doc['date_str'].split('/')[-1]})")
        print(f"Bradcast dates: {doc.get('first_broadcast_dates', ['N/A'])[0]}, Recording dates: {doc.get('recording_dates', ['N/A'])[0]}")
        print(f"MP3 filenames: {doc.get('mp3_filenames', ['N/A'])}")
        
        if doc['mp3_filenames']:
            for f in doc.get('mp3_filenames'):
                print(f" --> MP3 filename in audios: {f in audio_files}, OID in ASR files: {any(doc['stripped_OID'] in xml_f for xml_f in text_files)}")
        else:
            print(f"Doc {idx} has no mp3 filenames!! full doc:\n{doc}")
        #if doc.get('supports'):
        #    print(f"Audio file: {doc['supports'][0].get('filename', 'N/A')}")



Starting extracting 8 documents for program causerie_uni
Skipping document 3/8 with OID {1656BB3F-A8EF-4BF4-BE9A-20483F5665C8} because it's missing its XML file!! (stt exists: False)
Skipping document 5/8 with OID {87381313-87FD-4D31-AD19-8CDC08BA94DA} because it's missing its XML file!! (stt exists: False)
causerie_uni - returning 6 documents, 2 were skipped due to missing necessary info.
Total documents found: 6

Title: N/A...
Broadcast program name: Causerie universitaire
Extracted boradcast date: 19/12/1939 (year 1939)
Bradcast dates: 19/12/1939 - __/__/____, Recording dates: 20/11/1939 - 20/11/1939
MP3 filenames: ['3267f04d-657f-4dcb-bcb0-44b2b6c2682d_1211554131-1466X_complet_wav_958-SIROM{CFEC57B3-AADF-47BB-8ACF-49BE06EB6AD3}.mp3']
 --> MP3 filename in audios: True, OID in ASR files: True

Title: N/A...
Broadcast program name: Causerie universitaire
Extracted boradcast date: 07/01/1941 (year 1941)
Bradcast dates: 07/01/1941 - __/__/____, Recording dates: 26/11/1940 - 26/11/1940


In [10]:
all_documents[:5]

[{'alias': 'causerie_uni',
  'date_str': '19/12/1939',
  'mp3_filenames': ['3267f04d-657f-4dcb-bcb0-44b2b6c2682d_1211554131-1466X_complet_wav_958-SIROM{CFEC57B3-AADF-47BB-8ACF-49BE06EB6AD3}.mp3'],
  'cls_ID': '{D2593F4E-C887-4E48-8982-5BD08BA4DAE0}',
  'OID': '{3267F04D-657F-4DCB-BCB0-44B2B6C2682D}',
  'stripped_OID': '3267F04D-657F-4DCB-BCB0-44B2B6C2682D',
  'login': '',
  'broadcast_episode_title': "Adolphe Dygasinski, un Kipling avant Kipling. Causerie de Edouard Cros, privat-docent ? l'Universit? de Fribourg",
  'sequence': '1',
  'broadcast_program_name': 'Causerie universitaire',
  'document_type': 'Parl?',
  'hierarchy_level': 'Sujet',
  'modified_by': 'albrecjo',
  'modified_on': '25.05.2022 03:56:53',
  'physical_support_history': 'Disques 78T maison Radio-Lausanne 1466X A, 1467X A, 1466X B, 1467X B',
  'production_type': 'Production propre',
  'recording_place': 'Lausanne (Studio de Radio-Lausanne)',
  'rights_notes': 'Memoriav',
  'rights_status': 'Clarifi?',
  'series_title

In [171]:
doc_keys

dict_keys(['alias', 'date_str', 'mp3_filenames', 'cls_ID', 'OID', 'stripped_OID', 'login', 'broadcast_episode_title', 'sequence', 'broadcast_program_name', 'document_type', 'hierarchy_level', 'modified_by', 'modified_on', 'physical_support_history', 'production_type', 'recording_place', 'rights_notes', 'rights_status', 'series_title', 'content_summary', 'workflow_status', 'assembly_status', 'live', 'modilation_type', 'work_duration', 'work_duration_compl', 'participants', 'geographical_descriptors', 'person_descriptors', 'thematical_descriptors', 'rights_uage_possibilities', 'radio_channels', 'subdomains', 'recording_dates', 'first_broadcast_dates', 'supports', 'spt_filenames'])

In [11]:
doc_keys = all_documents[0].keys()

out_csv_name = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/RTS/debug_metadata.csv"
with open(out_csv_name, "w", newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, doc_keys)
    dict_writer.writeheader()
    dict_writer.writerows(all_documents)


This works and yields the desired set of metadata. 

## Aggregating all the programs' metadata into a single file

Now we need to write an orchestrator function that opens and processes the XML documents for each provider, and stores the info in a dict or a dataframe, also extracting the first braodcast date to help with the creation of the issue index file

In [12]:
all_program_aliases = sorted([p for p in os.listdir(base_dir) if "DS_Store" not in p and "$RECYCLE" not in p and "System" not in p])
print(f"Found {len(all_program_aliases)} Radio programs: {all_program_aliases}")

Found 47 Radio programs: ['ana_media', 'bigbang', 'canal_euro', 'causerie_uni', 'chron_instit', 'chron_unesco', 'courrier_cr', 'culte', 'dos_sci', 'ecoute_paix', 'enquest', 'forum', 'forum_lau', 'geneve_info', 'hist_ondes', 'infopile', 'inst_monde', 'j13h', 'j13h2', 'j_mat', 'j_midi', 'j_nuit', 'j_soir', 'mag_eco', 'mag_info', 'mag_sci1', 'mag_sci2', 'mag_tv1', 'mag_tv2', 'mem_ondes', 'min_oecu', 'miroir_monde', 'miroir_temps', 'monde_ant', 'monde_sem', 'nickel', 'nu_parle', 'ombres_eco', 'paraboles', 'paris_parle', 'parole_prem', 'petitdej', 'suisse_euro', 'terre_ciel', 'trib_prem', 'vie_monde', 'vie_va']


In [160]:
doc_keys.append('TEST')

In [ ]:
doc_keys = ['alias', 'date_str', 'mp3_filenames', 'cls_ID', 'OID', 'stripped_OID', 'login', 'broadcast_episode_title', 'sequence', 'broadcast_program_name', 'document_type', 'hierarchy_level', 'modified_by', 'modified_on', 'physical_support_history', 'production_type', 'recording_place', 'rights_notes', 'rights_status', 'series_title', 'content_summary', 'workflow_status', 'assembly_status', 'live', 'modilation_type', 'work_duration', 'work_duration_compl', 'participants', 'geographical_descriptors', 'person_descriptors', 'thematical_descriptors', 'rights_uage_possibilities', 'subdomains', 'recording_dates', 'supports']


In [21]:
out_csv_name = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/RTS/programs_metadata.rts.csv"

In [184]:
all_program_aliases.index("mem_ondes")

29

In [27]:
already_done = all_program_aliases[:all_program_aliases.index("mem_ondes")+1]
already_done = []

In [28]:
#already_done = all_program_aliases[:all_program_aliases.index("mem_ondes")+1]
#already_done = []
already_done

[]

In [ ]:
all_docs = []
all_docs_dict = {}
#doc_keys = None

for p_idx, p_alias in tqdm(enumerate(all_program_aliases)):

    if p_alias in already_done:
        print(f"\n{p_alias} is already done, skipping!")
        continue
    
    # first find all the metadata xml docs
    program_dir = os.path.join(base_dir, p_alias)
    metadata_files = [os.path.join(program_dir,f) for f in os.listdir(program_dir) if f.startswith(metadata_file_start)]
    
    all_audios_for_alias = os.listdir(os.path.join(program_dir,'audio'))
    all_stt_for_alias = os.listdir(os.path.join(program_dir,'stt'))

    print(f"\nPROCESSING PROGRAM {p_alias} ({p_idx+1}/{len(all_program_aliases)}) - {len(metadata_files)} files:")

    program_docs = []
    for xml_doc_path in tqdm(metadata_files):
        with open(xml_doc_path, "r", encoding="utf-8") as f:
            raw_xml = f.read()

        program_docs.extend(parse_docs_in_xml(BeautifulSoup(raw_xml, "xml"), p_alias, all_stt_for_alias, all_audios_for_alias, doc_keys))
    
    all_docs.extend(program_docs)
    all_docs_dict[p_alias] = program_docs

    # Save the current list of documents to save the progress
    #if not doc_keys:
    #    doc_keys = all_docs[0].keys()

    print(f" --> Adding {len(program_docs)} to the out csv for {p_alias}")
    with open(out_csv_name, "a", newline='') as output_file:
        dict_writer = csv.DictWriter(output_file, doc_keys)
        if not already_done:
            dict_writer.writeheader()
        dict_writer.writerows(all_docs)

    already_done.append(p_alias)
        


0it [00:00, ?it/s]


PROCESSING PROGRAM ana_media (1/47) - 1 files:


100%|██████████| 1/1 [00:00<00:00, 15.06it/s]
1it [00:00,  1.48it/s]


Starting extracting 6 documents for program ana_media
ana_media - returning 6 documents, 0 were skipped due to missing necessary info.
 --> Adding 6 to the out csv for ana_media

PROCESSING PROGRAM bigbang (2/47) - 1 files:



Starting extracting 40 documents for program bigbang
Skipping document 1/40 with OID {06660C0A-1C9C-4EEB-B9F0-9DD8977CA087} because it's missing its XML file!! (stt exists: False)
Skipping document 4/40 with OID {781626D5-C9EA-41F0-B495-42812C3C3F3F} because it's missing its XML file!! (stt exists: False)
Skipping document 6/40 with OID {42FA7346-699C-4F81-A4EF-0415C88960CD} because it's missing its XML file!! (stt exists: False)
Skipping document 9/40 with OID {D34C4835-E7BB-4C5A-9C3A-DE295A646F37} because it's missing its XML file!! (stt exists: False)
Skipping document 10/40 with OID {DB2CEE23-A884-42FD-94D5-2806D2A2553F} because it's missing its XML file!! (stt exists: False)
Skipping document 12/40 with OID {276678AF-E2C5-4E28-B471-08ABDBFF2789} because it's missing its XML file!! (stt exists: False)
Skipping document 14/40 with OID {797ED203-EDDC-4DBB-B3D9-B316EC0C9C43} because it's missing its XML file!! (stt exists: False)
Skipping document 16/40 with OID {64B8E4FD-527A-4769-9

...

Skipping document 92/95 with OID {4BB3BEBB-CC19-4C6D-9357-ED343C37C04F} because it's missing its XML file!! (stt exists: False)
dos_sci - returning 83 documents, 12 were skipped due to missing necessary info.
 --> Adding 83 to the out csv for dos_sci

PROCESSING PROGRAM ecoute_paix (10/47) - 1 files:


100%|██████████| 1/1 [00:00<00:00,  1.65it/s]
10it [00:10,  1.39it/s]


Starting extracting 125 documents for program ecoute_paix
Skipping document 1/125 with OID {9590C9BC-874D-4A91-84B9-89EF0B250D68} because it's missing its XML file!! (stt exists: False)
The date for document with OID {5BF47ED2-2B35-430F-B74B-12884F25292E} was invalid (~__/06/1947) - changed it to ~01/06/1947
The date for document with OID {BC0687B2-629E-4498-BD7E-592EDAD404A1} was invalid (~__/02/1947) - changed it to ~01/02/1947
The date for document with OID {BC0687B2-629E-4498-BD7E-592EDAD404A1} was invalid (~__/02/1947) - changed it to ~01/02/1947
Skipping document 4/125 with OID {10F4448C-EB47-486C-84D1-4E4269DA4AA3} because it's missing its XML file!! (stt exists: False)
Skipping document 5/125 with OID {EB75D4BA-E7A8-49EA-903E-9F876FBBF3B5} because it's missing its XML file!! (stt exists: False)
Skipping document 6/125 with OID {E5E5197A-3AB8-44F1-AA0C-A8F06EF7101D} because it's missing its XML file!! (stt exists: False)
Skipping document 7/125 with OID {C3BB5A60-4693-4A45-A4AD


Starting extracting 120 documents for program enquest
Skipping document 87/120 with OID {3055598F-87D5-4120-8F2D-F6570E49D3AB} because it's missing its XML file!! (stt exists: False)


Skipping document 94/120 with OID {59B38FA9-976E-4E14-87A8-172935248216} because it's missing its XML file!! (stt exists: False)
Skipping document 96/120 with OID {345F9ED0-471C-4C54-ADB9-3D6706F84EBF} because it's missing its XML file!! (stt exists: False)
The date for document with OID {18490C22-82F5-4AC3-9451-E1EFB2D9F100} was invalid (__/__/1979) - changed it to 01/01/1979
Skipping document 117/120 with OID {553708D4-F74C-4522-8275-CCB9E42B0F95} because it's missing its XML file!! (stt exists: False)
Skipping document 118/120 with OID {E00B1CB8-651F-414D-9896-5D6BEF5FACCD} because it's missing its XML file!! (stt exists: False)
enquest - returning 115 documents, 5 were skipped due to missing necessary info.


In [ ]:
metadata_df = pd.read_csv(out_csv_name)
metadata_df

## index_file building (not final nor correct)

In [70]:
# Function to build issue index structure similar to issue_index.sub.json
def build_issue_index(documents, program_alias, program_dir):
    """
    Build an issue index structure from parsed documents.
    
    Structure:
    {
        "program_alias": {
            "year": {
                "month": [
                    {
                        "day": int,
                        "edition": str (a, b, c, etc.),
                        "local_path": str,
                        "audio_file": str or list,
                        ... other document metadata
                    },
                    ...
                ],
                ...
            },
            ...
        }
    }
    
    Args:
        documents: list of parsed document dictionaries
        program_alias: the program name/alias (e.g., "causerie_uni")
        program_dir: the path to the program directory
    
    Returns:
        tuple: (issue_index_dict, missing_audio_files_dict)
    """
    from datetime import datetime
    from collections import defaultdict
    
    issue_index = {program_alias: {}}
    missing_audio_files = {program_alias: []}
    
    # Group documents by recording date
    date_groups = defaultdict(list)
    
    for doc in documents:
        # Parse recording date (format: "DD/MM/YYYY - DD/MM/YYYY")
        recording_dates = doc.get('recordingdates', [])
        if not recording_dates:
            missing_audio_files[program_alias].append({
                'title': doc.get('TITLE'),
                'reason': 'No recording date'
            })
            continue
        
        date_str = recording_dates[0].split(' - ')[0]  # Get first date
        try:
            date_obj = datetime.strptime(date_str, "%d/%m/%Y")
            year = str(date_obj.year)
            month = f"{date_obj.month:02d}"
            day = date_obj.day
            date_key = (year, month, day)
            date_groups[date_key].append(doc)
        except:
            missing_audio_files[program_alias].append({
                'title': doc.get('TITLE'),
                'reason': f'Could not parse date: {date_str}'
            })
            continue
    
    # Build the nested structure
    for (year, month, day), docs_for_day in sorted(date_groups.items()):
        # Initialize year and month if not exist
        if year not in issue_index[program_alias]:
            issue_index[program_alias][year] = {}
        if month not in issue_index[program_alias][year]:
            issue_index[program_alias][year][month] = []
        
        # Assign editions (a, b, c, etc.) to documents on the same day
        for edition_idx, doc in enumerate(docs_for_day):
            edition = chr(ord('a') + edition_idx)  # 'a', 'b', 'c', ...
            
            # Extract audio files
            audio_files = []
            if doc.get('supports'):
                for support in doc['supports']:
                    filename = support.get('filename')
                    if filename:
                        audio_files.append(filename)
            
            # Create issue entry
            issue_entry = {
                'day': day,
                'edition': edition,
                'local_path': program_dir,
                'title': doc.get('TITLE'),
                'broadcast': doc.get('BROADCAST'),
                'summary': doc.get('SUMMARY'),
                'participants': doc.get('participants', []),
                'geographic_descriptors': doc.get('geographicaldescriptors', []),
                'thematic_descriptors': doc.get('thematicaldescriptors', []),
                'duration': doc.get('WORKDURATION'),
                'recording_dates': doc.get('recordingdates', [])
            }
            
            # Handle audio_file field (single value if 1 file, list if multiple)
            if not audio_files:
                # Missing audio files - add to separate tracking dict
                missing_audio_files[program_alias].append({
                    'title': doc.get('TITLE'),
                    'date': f"{day}/{month}/{year}",
                    'edition': edition,
                    'reason': 'No audio files in metadata'
                })
            elif len(audio_files) == 1:
                issue_entry['audio_file'] = audio_files[0]
            else:
                issue_entry['audio_file'] = audio_files
            
            issue_index[program_alias][year][month].append(issue_entry)
    
    return issue_index, missing_audio_files


# Test with the example documents
issue_index, missing_files = build_issue_index(all_documents, 'causerie_uni', '/mnt/project_impresso/original/RTS/causerie_uni')

print(f"Issue Index built successfully!")
print(f"Programs: {list(issue_index.keys())}")

for program, years_data in issue_index.items():
    print(f"\n{program}:")
    print(f"  Years: {sorted(years_data.keys())}")
    
    for year, months_data in sorted(years_data.items()):
        print(f"    {year}:")
        for month, issues in sorted(months_data.items()):
            print(f"      {month}: {len(issues)} issue(s)")
            for issue in issues[:2]:  # Show first 2
                audio_info = f"audio: {issue.get('audio_file', 'N/A')}"
                if isinstance(issue.get('audio_file'), list):
                    audio_info = f"audio: {len(issue['audio_file'])} files"
                print(f"        Day {issue['day']}-{issue['edition']}: {audio_info}")

print(f"\n\nMissing audio files:")
for program, missing in missing_files.items():
    print(f"{program}: {len(missing)} document(s) with missing audio")
    for m in missing[:3]:  # Show first 3
        print(f"  - {m.get('title', 'N/A')[:60]}... ({m.get('reason', 'N/A')})")


Issue Index built successfully!
Programs: ['causerie_uni']

causerie_uni:
  Years: ['1939', '1940', '1942']
    1939:
      11: 6 issue(s)
        Day 4-a: audio: 1204286700-1542X_p_complet_wav958-SIROM{A3020B9E-9A94-4DFD-B1E0-CC34FE2F13AC}.wav
        Day 20-a: audio: 1211554131-1466X_complet_wav_958-SIROM{CFEC57B3-AADF-47BB-8ACF-49BE06EB6AD3}.wav
    1940:
      11: 1 issue(s)
        Day 26-a: audio: 1209129877-3151_p_complet_wav_958-SIROM{8357777C-7412-4674-98A8-B3AC6AB572DB}.wav
    1942:
      10: 1 issue(s)
        Day 3-a: audio: N/A


Missing audio files:
causerie_uni: 2 document(s) with missing audio
  - Le Tessin, facteur de coh?sion nationale : Causerie de Giova... (No audio files in metadata)
  - Epicure ou la religion du plaisir : Causerie de Ren? Schaere... (No audio files in metadata)


In [71]:
issue_index

{'causerie_uni': {'1939': {'11': [{'day': 4,
     'edition': 'a',
     'local_path': '/mnt/project_impresso/original/RTS/causerie_uni',
     'title': "L'enseignement de l'histoire. Causerie de Gonzague de Reynold",
     'broadcast': 'Causerie universitaire',
     'summary': "Cr?ation ? l'Universit? de Fribourg de la Chaire d'histoire de la civilisation moderne. L'enseignement de l'histoire a pour but d'initier les auditeurs ? la vie, de leur donner les connaissances et la compr?hension de l'Europe actuelle et tragique. L'?ducation de la pens?e doit d?velopper le sens critique en multipliant les termes de comparaison. La m?connaissance de l'histoire comme sympt?me de l'inculture d'une grande partie de la jeunesse. Devenir contemporain du pass? pour mieux le comprendre et appr?hender le pr?sent ? la lumi?re des origines.",
     'participants': [{'name': 'Reynold, Gonzague de',
       'function': 'Conf?rencier/e',
       'role': '?crivain'}],
     'geographic_descriptors': ['Fribourg (can